In [ ]:
import pandas as pd
import json
import os
import requests

<a id="Route-analyser"></a>
<h3 style="color:#1f77b4;">Route Analyser</h3>
<a id="Loading-Input-URLS"></a>
<h5 style="color:#1fb4a7;">Load route addresses</h5>

In [1]:
HOME_LAT = -31.863708969515947
HOME_LON = 116.04151607642865

ROUTE_CSV = r"C:\Users\thoma\Documents\balga_r_deadwalk.csv"

streets_to_find = [
    "Stedham Way",
    "Brede Place",
    "St Kilda Road",
    "Fitzroy Place",
    "Mentone Road",
    "Grinstead Way",
]

##### Get street geometry from Overpass

In [ ]:
cache_file = r"C:\Users\thoma\Documents\balga_streets.json"

# Reuse street data to prevent requesting many times
if os.path.exists(cache_file):
    with open(cache_file) as file:
        data = json.load(file)

else:
    # Overpass query for the selected Balga streets
    # NOTES FOR FUTURE:
    # [out:json] returns the result as JSON
    # area finds the Balga boundary and stores it as .a
    # way(area.a) searches for roads inside that boundary
    # ["highway"] keeps only road features
    # ["name"~"..."] matches any street name joined with |
    # out geom returns each road with its coordinates
    query = f"""
    [out:json];
    area["name"="Balga"]["admin_level"]->.a;
    way(area.a)["highway"]["name"~"{"|".join(streets_to_find)}"];
    out geom;
    """

    # Send Overpass API request
    response = requests.get(
        "https://overpass-api.de/api/interpreter",
        params={"data": query},
        timeout=100,
    )

    # Convert to JSON
    data = response.json()

    with open(cache_file, "w") as file:
        json.dump(data, file)

# Show number of returned elements
print(f"{len(data['elements'])} elements")